# 11 · Baselines

Tasks 16.3–16.5, walk-forward over the validation year 2023.

| Model | What it forecasts |
|---|---|
| `base_rate` | The training share of up moves, as a constant. Its direction is buy-and-hold's whenever that share is above 50%. |
| `ma_crossover` | Up while EMA 20 is above EMA 50 at the previous close, down otherwise; `p_up` is the training up-rate in each state. |
| `hist_vol` | The last 20 days' realised volatility, for every horizon. |

Buy-and-hold *returns* (with costs) come from the backtester, task 17.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.analysis.plots import (BLUE_CMAP, HORIZON_COLORS, MODEL_COLORS, TEXT_2, apply_style, bar_style,
                                dot_compare, figures_dir, save)
from src.config import Config
from src.evaluation.metrics import calibration_table, evaluate, join_labels
from src.models.data import label_frame, load_panel
from src.models.walkforward import run_or_load

cfg = Config.load()
apply_style()
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)
val_start, val_end = cfg.period("validation")

In [ ]:
RETRAIN = False          # True retrains even when saved forecasts match the current settings
panel = load_panel(cfg)
labels = label_frame(panel)
fc = pd.concat([run_or_load(m, cfg, "validation", panel=panel, retrain=RETRAIN)
                for m in ["base_rate", "ma_crossover", "hist_vol"]], ignore_index=True)
metrics = evaluate(fc, labels)
for m, g in metrics.groupby("model"):
    g.to_csv(figures_dir(cfg) / f"{m}_validation.csv", index=False)

## Direction

In [ ]:
direction = metrics[metrics.model != "hist_vol"]
direction[["model", "horizon_days", "n", "accuracy", "f1_up", "brier", "log_loss", "actual_up", "predicted_up"]]

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
dot_compare(ax, direction, "accuracy", ["ma_crossover", "base_rate"], panel.horizons, reference=0.5)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Direction accuracy, validation 2023")
save(fig, cfg, "11_baselines_accuracy")
plt.show()

## Volatility

`vol_mae` is in daily log-return units (0.005 = 0.5% a day).

In [ ]:
metrics[metrics.model == "hist_vol"][["model", "horizon_days", "n_vol", "vol_mae", "vol_rmse"]]

In [ ]:
j = join_labels(fc[(fc.model == "hist_vol") & (fc.horizon_days == 5)], labels)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
hb = ax.hexbin(j.vol_forecast, j.vol, gridsize=40, cmap=BLUE_CMAP, mincnt=1, bins="log", extent=(0, 0.05, 0, 0.05))
ax.plot([0, 0.05], [0, 0.05], color=TEXT_2, linewidth=1, linestyle="--")
fig.colorbar(hb, ax=ax, label="stock-days (log scale)")
ax.set_xlabel("forecast: last 20 days' volatility")
ax.set_ylabel("realised volatility over the next 5 days")
ax.set_title("Historical volatility vs what followed (5-day)")
save(fig, cfg, "11_hist_vol_scatter")
plt.show()

## MA crossover: up-rates learned from the training years

In [ ]:
import json
fold = sorted((cfg.models_dir / "ma_crossover" / "validation").glob("fold_*"))[0]
rates = json.loads((fold / "params.json").read_text())["rate"]
pd.DataFrame(rates).T.rename_axis("horizon")

## Findings

1. **Base rate:** 53.1% / 56.2% / 60.6% accuracy at 1 / 5 / 10 days in 2023, by predicting 'down' at 1 day and 'up' at 5 and 10 days. This is the bar for every other model.
2. **MA crossover** is below the base rate at every horizon: 49.7% / 53.3% / 54.4%. In training, stocks rose less often after EMA 20 crossed above EMA 50 than when it was below (50.8% vs 52.3% at 5 days; 52.3% vs 55.0% at 10 days), so the classic trend rule points the wrong way for these stocks and horizons.
3. **Historical volatility:** MAE 0.0088 / 0.0057 / 0.0050 (daily log-return units) at 1 / 5 / 10 days. The 1-day target is a single absolute return, so its error is the largest.